<a href="https://colab.research.google.com/github/VarvaraSharutina/bioinfo/blob/main/bioinf_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Математические методы биоинформатики

### Home work 4. Задача обучения с учителем - регрессия.

### Задание.

В файле *cardiovascular_hospital_los.csv* представлены биомедицинские данные госпитализированных пациентов с сердечно-сосудистыми заболеваниями. Необходимо предсказать длительность госпитализации (целевая переменная *'los_days'*) на основании представленных признаков.

Выполните следующие задания:

1) Проведите полный цикл предварительной обработки данных.  
- Обратите внимание, поскольку задача построения регрессионной модели - это задача обучения с учителем, то данные должны быть разделены на обучающую и тестовую выборки. Поэтому будьте внимательны и применяйте шаги предварительной обработки в правильном порядке.

2) Сформируйте три подмножества признаков: subset1 (все признаки), subset2 (статистически значимые признаки для модели линейной регрессии), subset3 (признаки, статистически значимо коррелирующие с целевой переменной).  
- Обратите внимание, что подмножества subset2 и subset3 должны быть получены на основе обучающей выборки, а не всего набора данных.  
- Также обратите внимание, что при построении subset3 для категориальных переменных нельзя напрямую использовать тесты на корреляцию. Вопрос в случае категориальных переменных формулируется как "Существуют ли отличия в длительности госпитализации между категориями?", поэтому необходимо подобрать нужный статистический тест. Также, например, если некоторый категориальный признак С имеет более двух категорий, то для статистического тестирования необходимо рассматривать непосредственно этот признак со всеми его категориями, а не отдельно каждый столбец из его One Hot Encoding версии. В случае, если статистическая значимость будет установлена, то в subset3 включаем уже One Hot Encoding столбцы этого признака.
- Наблюдаются ли отличия между полученными subset2 и subset3? Какие признаки есть в одном подмножестве, но нет в другом?

3) Примените модель линейной регрессии LinearRegression к трем наборам признаков: subset1, subset2 и subset3. Вычислите следующие метрики качества на тестовой выборке: скорректированный коэффициент детерминации, RMSE, MAE. Наблюдается ли улучшение качества предсказаний при рассмотрении subset2 и subset3 по сравнению с subset1?

4) Примените модель ElasticNet к набору признаков subset1. Для этого выполните подбор двух гиперпараметров модели с помощью кросс-валидации по 5 блокам. Вычислите те же метрики качества результирующей оптимальной модели (с подобранными оптимальными гиперпараметрыми) на тестовой выборке. Наблюдается ли улучшение качества по сравнению с моделями из предыдущего пункта?

In [280]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy.stats import spearmanr, mannwhitneyu, kruskal
import sklearn
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

url = "https://raw.githubusercontent.com/VarvaraSharutina/bioinfo/main/cardiovascular_hospital_los.csv"
data_raw = pd.read_csv(url, sep='\t')

In [281]:
data_raw

,age,gender,smoking,alcohol_last_week,sbp,dbp,heart_rate,severity_score,creatinine,troponin,...,platelets,ldl,hdl,glucose,tnfa_expression,il6_expression,mmp9_expression,vegf_expression,pcsk9_mutation,los_days
0,70.0,Male,1,0,113,92,123,119,3.695930,3.522529,...,272.0,4.230000,1.260000,4.4,2.72,34.99,41.03,55.11,Wild,20.0
1,63.0,Male,0,0,119,74,70,125,1.590031,0.596930,...,346.0,1.851611,1.030000,8.4,9.44,8.72,9.16,320.70,Homozygous,12.0
2,72.0,Male,1,1,220,88,51,134,3.301389,1.211761,...,270.0,3.060000,0.704008,5.9,11.49,4.66,11.25,41.92,Wild,17.0
3,83.0,Female,0,1,178,57,76,121,3.737128,15.982889,...,307.0,3.010000,2.500000,6.3,13.21,17.56,39.74,152.04,Wild,22.0
4,62.0,Male,0,0,106,94,45,124,5.000000,0.728508,...,327.0,2.770000,0.730000,7.2,9.77,4.04,43.70,93.92,Wild,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
738,84.0,Male,0,1,99,89,89,124,2.751676,23.392335,...,227.0,3.870000,1.601818,5.6,10.49,14.02,16.95,195.85,Wild,18.0
739,77.0,Male,0,0,166,54,75,103,1.810306,50.000000,...,247.0,2.464681,1.210000,8.0,5.99,6.32,29.20,320.18,Heterozygous,22.0
740,42.0,Male,0,0,136,93,117,128,3.116542,1.235204,...,251.0,5.870000,1.310000,7.9,20.80,30.19,50.11,130.97,Wild,13.0
741,49.0,Male,0,0,159,77,85,114,1.523342,2.252218,...,263.0,1.431739,0.960000,6.0,5.85,5.34,5.36,56.38,Homozygous,15.0


1. Проведите полный цикл предварительной обработки данных.

In [282]:
# Поиск неявных дубликатов
print(data_raw['gender'].value_counts())
print("-" * 40)
print(data_raw['smoking'].value_counts())
print("-" * 40)
print(data_raw['alcohol_last_week'].value_counts())
print("-" * 40)
print(data_raw['pcsk9_mutation'].value_counts())
print("-" * 40)

data_raw = data_raw[data_raw['alcohol_last_week'] != 2]
print(data_raw['alcohol_last_week'].value_counts())

gender
Male      391
Female    351
Name: count, dtype: int64
----------------------------------------
smoking
0    442
1    301
Name: count, dtype: int64
----------------------------------------
alcohol_last_week
0    508
1    233
2      2
Name: count, dtype: int64
----------------------------------------
pcsk9_mutation
Wild            531
Heterozygous    168
Homozygous       44
Name: count, dtype: int64
----------------------------------------
alcohol_last_week
0    508
1    233
Name: count, dtype: int64


In [283]:
# Поиск полных дубликатов
duplicates = data_raw.duplicated().sum()
print(f"Количество полных дубликатов: {duplicates}")

Количество полных дубликатов: 0


In [284]:
# Обработка пропусков
data_raw.isnull().sum()

,0
age,1
gender,1
smoking,0
alcohol_last_week,0
sbp,0
dbp,0
heart_rate,0
severity_score,0
creatinine,0
troponin,1


In [285]:
data_raw=data_raw.dropna()
data_raw.isnull().sum()

,0
age,0
gender,0
smoking,0
alcohol_last_week,0
sbp,0
dbp,0
heart_rate,0
severity_score,0
creatinine,0
troponin,0


In [286]:
# Обработка категориальных признаков
data_raw['gender'] = data_raw['gender'].map({'Male': 1,'Female': 0})
data_raw_copy=data_raw
data_raw = pd.get_dummies(data_raw, columns=['pcsk9_mutation'])
# Разделение на тестовую и обучающую выборки
df=data_raw
X=df.drop('los_days', axis=1)
y=df['los_days'] # целевая переменная
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

# Анализ выбросов
numeric_cols=['age', 'sbp',	'dbp',	'heart_rate',	'severity_score',	'creatinine',	'troponin',	'ck_mb',	'nt_probnp',	'crp',	'wbc',
              'hb',	'platelets',	'ldl',	'hdl',	'glucose',	'tnfa_expression',	'il6_expression',	'mmp9_expression',	'vegf_expression']
categorial_cols=['gender', 'smoking', 'alcohol_last_week', 'pcsk9_mutation']
cat_cols=['gender', 'smoking', 'alcohol_last_week', 'pcsk9_mutation_Heterozygous',	'pcsk9_mutation_Homozygous',	'pcsk9_mutation_Wild']

def analysis_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    mini = Q1 - 1.5 * IQR
    maxi = Q3 + 1.5 * IQR
    outliers = data[(data[column] < mini) | (data[column] > maxi)]
    return outliers, mini, maxi

for col in numeric_cols:
    outliers, min, max = analysis_outliers(df, col)
    print(f"\n{col}")
    print(f"Границы: [{min:.2f}, {max:.2f}]")
    print(f"Количество выбросов: {len(outliers)} ")
    if len(outliers) > 0:
        print(f"Минимальное значение выброса: {outliers[col].min()}")
        print(f"Максимальное значение выброса: {outliers[col].max()}")
        print(f"Bыбросы: {outliers[col].tolist()}")


age
Границы: [32.00, 96.00]
Количество выбросов: 1 
Минимальное значение выброса: 30.0
Максимальное значение выброса: 30.0
Bыбросы: [30.0]

sbp
Границы: [74.12, 207.12]
Количество выбросов: 4 
Минимальное значение выброса: 215
Максимальное значение выброса: 220
Bыбросы: [220, 215, 220, 216]

dbp
Границы: [42.50, 126.50]
Количество выбросов: 2 
Минимальное значение выброса: 130
Максимальное значение выброса: 130
Bыбросы: [130, 130]

heart_rate
Границы: [32.38, 137.38]
Количество выбросов: 1 
Минимальное значение выброса: 141
Максимальное значение выброса: 141
Bыбросы: [141]

severity_score
Границы: [90.62, 147.62]
Количество выбросов: 3 
Минимальное значение выброса: 149
Максимальное значение выброса: 152
Bыбросы: [151, 152, 149]

creatinine
Границы: [-0.40, 6.46]
Количество выбросов: 2 
Минимальное значение выброса: 9.58842271125387
Максимальное значение выброса: 10.304068690278724
Bыбросы: [9.58842271125387, 10.304068690278724]

troponin
Границы: [-41.66, 80.12]
Количество выбросов: 

Оставим выбросы без обработки, так как их недостаточно много для удаления конкретных признаков, но если удалять образцы, то мы потеряем много информации

In [287]:
# z-нормализация
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Импутацию не проводим, так как удалили все пропуски (всего было 5 образцов с ними)

2. Сформируйте три подмножества признаков: subset1 (все признаки), subset2 (статистически значимые признаки для модели линейной регрессии), subset3 (признаки, статистически значимо коррелирующие с целевой переменной).

In [288]:
X_train_scaled_df = pd.DataFrame(X_train_scaled,columns=X_train.columns,index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled,columns=X_test.columns,index=X_test.index)

subset1 = numeric_cols +cat_cols

X_train_const = sm.add_constant(X_train_scaled_df)
ols_model = sm.OLS(y_train, X_train_const).fit()
p_values = ols_model.pvalues.drop('const')
subset2 = p_values[p_values < 0.05].index.tolist()

subset3=[]
X_train_orig = data_raw_copy.drop(columns=['los_days']).loc[X_train.index]
for col in numeric_cols:
  statistic, pvalue = stats.spearmanr(X_train[col], y_train)
  if pvalue < 0.05:
    subset3.append(col)

for col in categorial_cols:
    unique_vals = X_train_orig[col].unique()
    if len(unique_vals) == 2:
        group1 = y_train[X_train[col]==unique_vals[0]]
        group2 = y_train[X_train[col]==unique_vals[1]]
        stat, p_val = mannwhitneyu(group1, group2, alternative='two-sided')
    else:
        groups = [y_train[X_train_orig[col] == val] for val in unique_vals]
        stat, p_val = kruskal(*groups)
    if p_val < 0.05:
        subset3.append(col)

print(subset3)

['age', 'sbp', 'severity_score', 'creatinine', 'troponin', 'nt_probnp', 'crp', 'wbc', 'smoking']


3. Примените модель линейной регрессии LinearRegression к трем наборам признаков: subset1, subset2 и subset3. Вычислите следующие метрики качества на тестовой выборке: скорректированный коэффициент детерминации, RMSE, MAE. Наблюдается ли улучшение качества предсказаний при рассмотрении subset2 и subset3 по сравнению с subset1?

In [289]:
def lin_reg (X_train, X_test, y_train, y_test):
  lr_model=LinearRegression().fit(X_train, y_train)
  y_pred = lr_model.predict(X_test)

  R2 = r2_score(y_test, y_pred)
  n = len(y_test)
  d = X_train.shape[1]
  R2_adj = 1 - (1 - R2)*(n-1) / (n - d - 1)
  RMSE = root_mean_squared_error(y_test, y_pred)
  MAE = mean_absolute_error(y_test, y_pred)
  print(f"  Количество признаков: {d}")
  print(f"  R² (коэффициент детерминации): {R2:.4f}")
  print(f"  Скорректированный R²: {R2_adj:.4f}")
  print(f"  RMSE: {RMSE:.2f}")
  print(f"  MAE: {MAE:.2f}")

print("\nLinearRegression:")
print('\nsubset1:')
lin_reg(X_train_scaled_df[subset1], X_test_scaled_df[subset1], y_train, y_test)
print('\nsubset2:')
lin_reg(X_train_scaled_df[subset2], X_test_scaled_df[subset2], y_train, y_test)
print('\nsubset3:')
lin_reg(X_train_scaled_df[subset3], X_test_scaled_df[subset3], y_train, y_test)


LinearRegression:

subset1:
  Количество признаков: 26
  R² (коэффициент детерминации): 0.8237
  Скорректированный R²: 0.7858
  RMSE: 1.38
  MAE: 1.16

subset2:
  Количество признаков: 17
  R² (коэффициент детерминации): 0.8284
  Скорректированный R²: 0.8060
  RMSE: 1.36
  MAE: 1.13

subset3:
  Количество признаков: 9
  R² (коэффициент детерминации): 0.7767
  Скорректированный R²: 0.7621
  RMSE: 1.55
  MAE: 1.26


Самые лучшие результаты показала себя линейная регрессия на наборе признаков subset2, которые как раз-таки представляют из себя статистически значимые признаки для модели линейной регрессии.

Результат хуже можно наблюдать на subset1 и самые худшие показатели метрик на subset3.

4. Примените модель ElasticNet к набору признаков subset1. Для этого выполните подбор двух гиперпараметров модели с помощью кросс-валидации по 5 блокам. Вычислите те же метрики качества результирующей оптимальной модели (с подобранными оптимальными гиперпараметрыми) на тестовой выборке. Наблюдается ли улучшение качества по сравнению с моделями из предыдущего пункта?

In [291]:
param_grid = {'alpha': [0.01, 0.1, 1, 5, 10],
              'l1_ratio': [0.1, 0.5, 0.9, 1]}

grid_search = GridSearchCV(estimator=elastic,param_grid=param_grid,cv=5,scoring='neg_mean_squared_error',n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)
a = grid_search.best_params_['alpha']
l1 = grid_search.best_params_['l1_ratio']
print(grid_search.best_params_)

enet_model = ElasticNet(alpha=a, l1_ratio=l1, random_state=42)
enet_model.fit(X_train_scaled, y_train)
y_pred_en = best_model.predict(X_test_scaled)

R2 = r2_score(y_test, y_pred_en)
n = len(y_test)
d = X_train_scaled.shape[1]
R2_adj = 1 - (1 - R2) * (n - 1) / (n - d - 1)
RMSE = root_mean_squared_error(y_test, y_pred_en)
MAE = mean_absolute_error(y_test, y_pred_en)
print("\nElasticNet:")
print(f"  Количество признаков: {d}")
print(f"  R²: {R2:.4f}")
print(f"  Скорректированный R²: {R2_adj:.4f}")
print(f"  RMSE: {RMSE:.2f}")
print(f"  MAE: {MAE:.2f}")

{'alpha': 0.01, 'l1_ratio': 1}

ElasticNet:
  Количество признаков: 26
  R²: 0.8260
  Скорректированный R²: 0.7886
  RMSE: 1.37
  MAE: 1.15


Результаты ElasticNet на subset1 стали чуть лучше по сравнению с линейной регрессией на том же наборе признаков, но они не превзошли результатов линейной регрессии на subset2.